# 5 Modelling Click Screening v1

## 5.0 Setup

In [23]:
# basic imports
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [24]:
# sklearn imports
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss
)

In [25]:
# project paths
PROJECT_ROOT = Path.home() / "Documents" / "Thesis"
DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "Processed Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"

CLICK_MODEL_PATH = PROCESSED_DATA_DIR / "df_click_model_v1.parquet"

In [26]:
# modelling constants
TARGET_COL = "click"
GROUP_COL = "mailing_id"

FINAL_TEST_SIZE = 0.20
INNER_CV_SPLITS = 5

In [27]:
# quick path checks
print("Project root:", PROJECT_ROOT)
print("Processed data folder exists:", PROCESSED_DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Click model file exists:", CLICK_MODEL_PATH.exists())

Project root: /Users/khanhhien/Documents/Thesis
Processed data folder exists: True
Output folder exists: True
Click model file exists: True


## 5.1 Load click dataset

In [28]:
# load click modelling dataset
df_click = pd.read_parquet(CLICK_MODEL_PATH)

print("df_click shape:", df_click.shape)
df_click.head()

df_click shape: (1104242, 38)


,user_id,mailing_id,open,click,mailing_info,subject_line,preheader,send_date,send_day_of_week,date_trusted,...,preheader_contains_exclamation,prior_email_count,prior_open_count,prior_click_count,historical_open_rate,historical_click_rate,historical_open_bucket,prior_email_frequency_bucket,had_prior_click,is_first_email
0,5c6bebde5798a794283224c9,668,0.0,0.0,2025/04/24 CPX - MM MAX Magazine - 4 gratis nrs,"Lezen, puzzelen, genieten: 4 edities cadeau","Vraag nu aan – geen kosten, geen verplichtingen.",2025-04-25,Vrijdag,True,...,False,0,0.0,0.0,NaN,NaN,NaN,very_little,False,True
1,5c6bebde5798a794283224c9,691,0.0,0.0,2025/05/29 WOONCOMFORT AIRCO,Bespaar tot 80% op je energiekosten met een airco,Vraag nu gratis maatwerkadvies aan voor energi...,2025-04-30,Woensdag,True,...,False,1,0.0,0.0,0.0,0.0,very_low,very_little,False,False
2,5c6bebde5798a794283224c9,714,0.0,0.0,2025/05/06 CPX - MM - ENGIE BONUS,Profiteer nu: €300 bonus én vaste energietarieven,ENGIE helpt je graag met persoonlijk advies.,2025-05-07,Woensdag,True,...,False,2,0.0,0.0,0.0,0.0,very_low,very_little,False,False
3,5c6bebde5798a794283224c9,733,0.0,0.0,2025/05/14 LIFEPOINTS,Jouw mening is geld waard – start vandaag nog,Verdien punten voor digitale cadeaubonnen zoal...,2025-05-14,Woensdag,True,...,False,3,0.0,0.0,0.0,0.0,very_low,very_little,False,False
4,5c6bebde5798a794283224c9,761,0.0,0.0,2025/05/27 CPX - MM - ENGIE,Speel mee met de ENGIE woordlegger!,"En win een solar powerbank t.w.v. €79,95.",2025-06-04,Woensdag,True,...,False,4,0.0,0.0,0.0,0.0,very_low,very_little,False,False


In [29]:
# basic target checks
print("Target column:", TARGET_COL)
print("Missing target values:", df_click[TARGET_COL].isna().sum())

display(df_click[TARGET_COL].value_counts(dropna=False))
display(df_click[TARGET_COL].value_counts(normalize=True, dropna=False))

Target column: click
Missing target values: 0


click
0.0    1103047
1.0       1195
Name: count, dtype: int64

click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64

In [30]:
# campaign checks
print("Number of unique campaigns:", df_click[GROUP_COL].nunique())
print("Campaign ID range:", df_click[GROUP_COL].min(), "to", df_click[GROUP_COL].max())

display(df_click[GROUP_COL].describe())

Number of unique campaigns: 274
Campaign ID range: 653 to 1418


count      1104242.0
mean     1054.298506
std       223.344151
min            653.0
25%            874.0
50%           1031.0
75%           1274.0
max           1418.0
Name: mailing_id, dtype: Float64

In [31]:
# column overview
print("Number of columns:", df_click.shape[1])

click_column_overview = pd.DataFrame({
    "column": df_click.columns,
    "dtype": [df_click[col].dtype for col in df_click.columns],
    "missing_rate": [df_click[col].isna().mean() for col in df_click.columns]
}).sort_values("missing_rate", ascending=False)

display(click_column_overview)

Number of columns: 38


,column,dtype,missing_rate
13,age_group,category,0.744601
12,age_clean,float64,0.744601
8,send_day_of_week,object,0.325884
7,send_date,datetime64[us],0.325884
32,historical_open_rate,float64,0.099916
34,historical_open_bucket,category,0.099916
30,prior_open_count,float64,0.084261
2,open,float64,0.084261
6,preheader,object,0.029575
5,subject_line,object,0.024971


In [32]:
# verify critical columns
critical_cols = [
    "mailing_id",
    "click",
    "open",
    "campaign_topic",
    "gender",
    "age_group",
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "had_prior_click"
]

for col in critical_cols:
    print(f"{col}: {col in df_click.columns}")

mailing_id: True
click: True
open: True
campaign_topic: True
gender: True
age_group: True
historical_open_bucket: True
prior_email_frequency_bucket: True
had_prior_click: True


## 5.2 Recreate click feature sets and preprocessing design

In [33]:
# define click feature type dictionaries
feature_types_A = {
    "numeric": [
        "n_interests"
    ],

    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    ],

    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_contains_number",
        "subject_contains_promotion",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "had_prior_click",
        "is_first_email"
    ]
}

feature_types_B = {
    "numeric": feature_types_A["numeric"].copy(),

    "categorical": (
        feature_types_A["categorical"].copy()
        + [
            "campaign_topic_x_age_group",
            "historical_open_bucket_x_prior_email_frequency_bucket"
        ]
    ),

    "binary": feature_types_A["binary"].copy()
}

feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate"
    ],

    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    ],

    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_missing",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "is_first_email"
    ]
}

In [34]:
# derive feature sets from feature type dictionaries
def flatten_feature_types(feature_types):
    return (
        feature_types["numeric"]
        + feature_types["categorical"]
        + feature_types["binary"]
    )

feature_set_A_core = flatten_feature_types(feature_types_A)
feature_set_B_interactions = flatten_feature_types(feature_types_B)
feature_set_C_expanded = flatten_feature_types(feature_types_C)

feature_sets = {
    "A_core": feature_set_A_core,
    "B_interactions": feature_set_B_interactions,
    "C_expanded": feature_set_C_expanded
}

feature_type_sets = {
    "A_core": feature_types_A,
    "B_interactions": feature_types_B,
    "C_expanded": feature_types_C
}

for name, features in feature_sets.items():
    print(name, ":", len(features), "features")

A_core : 16 features
B_interactions : 18 features
C_expanded : 28 features


In [35]:
# create deterministic interaction columns
def combine_interaction_cols(df, col1, col2):
    return (
        df[col1].astype("object").fillna("Missing").astype(str)
        + " x "
        + df[col2].astype("object").fillna("Missing").astype(str)
    )

df_click["campaign_topic_x_age_group"] = combine_interaction_cols(
    df_click, "campaign_topic", "age_group"
)

df_click["historical_open_bucket_x_prior_email_frequency_bucket"] = combine_interaction_cols(
    df_click, "historical_open_bucket", "prior_email_frequency_bucket"
)

interaction_cols = [
    "campaign_topic_x_age_group",
    "historical_open_bucket_x_prior_email_frequency_bucket"
]

for col in interaction_cols:
    print(col)
    print("  exists:", col in df_click.columns)
    print("  missing rate:", df_click[col].isna().mean())
    print("  unique values:", df_click[col].nunique())

campaign_topic_x_age_group
  exists: True
  missing rate: 0.0
  unique values: 60
historical_open_bucket_x_prior_email_frequency_bucket
  exists: True
  missing rate: 0.0
  unique values: 30


In [36]:
# validate feature sets
def validate_feature_set(name, features, df):
    missing_features = [col for col in features if col not in df.columns]
    duplicated_features = pd.Series(features)[pd.Series(features).duplicated()].tolist()
    
    print("=" * 60)
    print(name)
    print("Number of features:", len(features))
    print("Missing features:", missing_features)
    print("Duplicated features:", duplicated_features)

for name, features in feature_sets.items():
    validate_feature_set(name, features, df_click)

A_core
Number of features: 16
Missing features: []
Duplicated features: []
B_interactions
Number of features: 18
Missing features: []
Duplicated features: []
C_expanded
Number of features: 28
Missing features: []
Duplicated features: []


In [37]:
# leakage and excluded-column check
forbidden_feature_cols = [
    "click",
    "open",
    "user_id",
    "mailing_id",
    "mailing_info",
    "subject_line",
    "preheader",
    "send_date",
    "send_day_of_week",
    "date_trusted"
]

for name, features in feature_sets.items():
    forbidden_found = [col for col in features if col in forbidden_feature_cols]
    print(name, "forbidden columns found:", forbidden_found)

A_core forbidden columns found: []
B_interactions forbidden columns found: []
C_expanded forbidden columns found: []


In [38]:
# build preprocessing recipes
def build_preprocessor(feature_types):
    numeric_features = feature_types["numeric"]
    categorical_features = feature_types["categorical"]
    binary_features = feature_types["binary"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
            ("binary", binary_transformer, binary_features)
        ],
        remainder="drop"
    )

    return preprocessor


preprocessors = {
    name: build_preprocessor(feature_types)
    for name, feature_types in feature_type_sets.items()
}

preprocessors.keys()

dict_keys(['A_core', 'B_interactions', 'C_expanded'])

## 5.3 Recreate temporal campaign holdout

In [39]:
# define chronological campaign holdout
campaign_ids_sorted = np.sort(df_click[GROUP_COL].dropna().unique())

n_campaigns = len(campaign_ids_sorted)
n_test_campaigns = int(np.ceil(n_campaigns * FINAL_TEST_SIZE))
n_trainval_campaigns = n_campaigns - n_test_campaigns

trainval_campaigns = campaign_ids_sorted[:n_trainval_campaigns]
test_campaigns = campaign_ids_sorted[n_trainval_campaigns:]

print("Total campaigns:", n_campaigns)
print("Train-validation campaigns:", len(trainval_campaigns))
print("Final test campaigns:", len(test_campaigns))

print("Trainval campaign range:", trainval_campaigns.min(), "to", trainval_campaigns.max())
print("Test campaign range:", test_campaigns.min(), "to", test_campaigns.max())

Total campaigns: 274
Train-validation campaigns: 219
Final test campaigns: 55
Trainval campaign range: 653 to 1252
Test campaign range: 1257 to 1418


In [40]:
# create train-validation and final test dataframes
trainval_df = df_click[df_click[GROUP_COL].isin(trainval_campaigns)].copy()
test_df = df_click[df_click[GROUP_COL].isin(test_campaigns)].copy()

print("Train-validation shape:", trainval_df.shape)
print("Final test shape:", test_df.shape)

Train-validation shape: (814269, 40)
Final test shape: (289973, 40)


In [41]:
# check campaign overlap
trainval_campaign_set = set(trainval_df[GROUP_COL].unique())
test_campaign_set = set(test_df[GROUP_COL].unique())

campaign_overlap = trainval_campaign_set.intersection(test_campaign_set)

print("Campaign overlap:", len(campaign_overlap))
print("Overlap values:", sorted(campaign_overlap)[:10])

assert len(campaign_overlap) == 0, "Leakage: campaign appears in both trainval and test."

Campaign overlap: 0
Overlap values: []


In [42]:
# check target distribution by split
split_summary = pd.DataFrame({
    "split": ["trainval", "test"],
    "n_rows": [len(trainval_df), len(test_df)],
    "n_campaigns": [
        trainval_df[GROUP_COL].nunique(),
        test_df[GROUP_COL].nunique()
    ],
    "click_rate": [
        trainval_df[TARGET_COL].mean(),
        test_df[TARGET_COL].mean()
    ],
    "click_positive_count": [
        int(trainval_df[TARGET_COL].sum()),
        int(test_df[TARGET_COL].sum())
    ],
    "click_negative_count": [
        int((trainval_df[TARGET_COL] == 0).sum()),
        int((test_df[TARGET_COL] == 0).sum())
    ]
})

display(split_summary)

,split,n_rows,n_campaigns,click_rate,click_positive_count,click_negative_count
0,trainval,814269,219,0.001228,1000,813269
1,test,289973,55,0.000672,195,289778


In [43]:
# store split summary
split_summary_click = split_summary.copy()
split_summary_click

,split,n_rows,n_campaigns,click_rate,click_positive_count,click_negative_count
0,trainval,814269,219,0.001228,1000,813269
1,test,289973,55,0.000672,195,289778


## 5.4 Recreate interaction columns

Already created in 5.2

## 5.5 Set up grouped cross-validation

In [44]:
# create reference X, y, groups for CV
X_reference = trainval_df[feature_set_A_core].copy()
y_reference = trainval_df[TARGET_COL].copy()
groups_reference = trainval_df[GROUP_COL].copy()

print("X_reference shape:", X_reference.shape)
print("y_reference shape:", y_reference.shape)
print("Number of groups:", groups_reference.nunique())
print("Click rate:", y_reference.mean())
print("Positive clicks:", int(y_reference.sum()))

X_reference shape: (814269, 16)
y_reference shape: (814269,)
Number of groups: 219
Click rate: 0.001228095383712262
Positive clicks: 1000


In [45]:
# create grouped stratified CV object
cv_strategy = StratifiedGroupKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_strategy

StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)

In [46]:
# check inner CV fold quality
fold_summaries = []

for fold_idx, (train_idx, val_idx) in enumerate(
    cv_strategy.split(X_reference, y_reference, groups_reference),
    start=1
):
    y_train_fold = y_reference.iloc[train_idx]
    y_val_fold = y_reference.iloc[val_idx]
    
    groups_train_fold = groups_reference.iloc[train_idx]
    groups_val_fold = groups_reference.iloc[val_idx]
    
    campaign_overlap = set(groups_train_fold).intersection(set(groups_val_fold))
    
    fold_summaries.append({
        "fold": fold_idx,
        "train_rows": len(train_idx),
        "val_rows": len(val_idx),
        "train_campaigns": groups_train_fold.nunique(),
        "val_campaigns": groups_val_fold.nunique(),
        "campaign_overlap": len(campaign_overlap),
        "train_click_rate": y_train_fold.mean(),
        "val_click_rate": y_val_fold.mean(),
        "val_click_positive_count": int(y_val_fold.sum()),
        "val_click_negative_count": int((y_val_fold == 0).sum())
    })

cv_fold_summary_click = pd.DataFrame(fold_summaries)

display(cv_fold_summary_click)

,fold,train_rows,val_rows,train_campaigns,val_campaigns,campaign_overlap,train_click_rate,val_click_rate,val_click_positive_count,val_click_negative_count
0,1,650238,164031,176,43,0,0.001143,0.001567,257,163774
1,2,650469,163800,175,44,0,0.001314,0.000885,145,163655
2,3,633122,181147,175,44,0,0.001395,0.000646,117,181030
3,4,666325,147944,176,43,0,0.001108,0.001771,262,147682
4,5,656922,157347,174,45,0,0.001189,0.001392,219,157128


In [47]:
# assert fold safety
assert (cv_fold_summary_click["campaign_overlap"] == 0).all(), "Campaign leakage inside CV folds."
assert (cv_fold_summary_click["val_click_positive_count"] > 0).all(), "A validation fold has no positive click cases."
assert (cv_fold_summary_click["val_click_negative_count"] > 0).all(), "A validation fold has no negative click cases."

print("CV fold safety checks passed.")

CV fold safety checks passed.


## 5.6 Define click metrics

In [48]:
# define metric calculation function
def calculate_binary_metrics(y_true, y_pred, y_proba):
    """
    Calculate binary classification metrics for one validation fold.
    
    For click prediction:
    - PR_AUC is the main metric because click is extremely rare.
    - ROC_AUC is secondary.
    - Accuracy is only an audit metric.
    """
    metrics = {
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "PR_AUC": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "brier": brier_score_loss(y_true, y_proba),
        "log_loss": log_loss(y_true, y_proba, labels=[0, 1])
    }
    
    return metrics

In [49]:
# test metric function on fake predictions
y_fake = pd.Series([0, 0, 1, 1])
pred_fake = pd.Series([0, 0, 1, 1])
proba_fake = pd.Series([0.1, 0.2, 0.8, 0.9])

calculate_binary_metrics(y_fake, pred_fake, proba_fake)

{'ROC_AUC': 1.0,
 'PR_AUC': 1.0,
 'accuracy': 1.0,
 'balanced_accuracy': 1.0,
 'brier': 0.024999999999999998,
 'log_loss': 0.16425203348601802}

In [50]:
# define metric column order
metric_names = [
    "ROC_AUC",
    "PR_AUC",
    "accuracy",
    "balanced_accuracy",
    "brier",
    "log_loss"
]

primary_metric = "PR_AUC"

metric_names

['ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'brier', 'log_loss']

## 5.7 Run dummy baselines

In [51]:
# cross-validation evaluation helper
def evaluate_cv_model(
    model,
    X,
    y,
    groups,
    cv_strategy,
    metric_names
):
    
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(
        cv_strategy.split(X, y, groups),
        start=1
    ):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]

        metrics = calculate_binary_metrics(
            y_true=y_val,
            y_pred=y_pred,
            y_proba=y_proba
        )

        metrics["fold"] = fold_idx

        fold_results.append(metrics)

    fold_results_df = pd.DataFrame(fold_results)

    return fold_results_df

In [52]:
# prepare X, y, groups for dummy baselines
X_A = trainval_df[feature_set_A_core]
y_A = trainval_df[TARGET_COL]
groups_A = trainval_df[GROUP_COL]

print("X_A shape:", X_A.shape)
print("Click rate:", y_A.mean())
print("Positive clicks:", int(y_A.sum()))

X_A shape: (814269, 16)
Click rate: 0.001228095383712262
Positive clicks: 1000


### 5.7.1 Majority-class dummy baseline

In [53]:
# majority dummy baseline
majority_dummy = DummyClassifier(
    strategy="most_frequent"
)

majority_cv_results = evaluate_cv_model(
    model=majority_dummy,
    X=X_A,
    y=y_A,
    groups=groups_A,
    cv_strategy=cv_strategy,
    metric_names=metric_names
)

display(majority_cv_results)

,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss,fold
0,0.5,0.001567,0.998433,0.5,0.001567,0.056472,1
1,0.5,0.000885,0.999115,0.5,0.000885,0.031907,2
2,0.5,0.000646,0.999354,0.5,0.000646,0.023280,3
3,0.5,0.001771,0.998229,0.5,0.001771,0.063831,4
4,0.5,0.001392,0.998608,0.5,0.001392,0.050167,5


### 5.7.2 Prior probability dummy baseline

In [54]:
# prior probability dummy baseline
prior_dummy = DummyClassifier(
    strategy="prior"
)

prior_cv_results = evaluate_cv_model(
    model=prior_dummy,
    X=X_A,
    y=y_A,
    groups=groups_A,
    cv_strategy=cv_strategy,
    metric_names=metric_names
)

display(prior_cv_results)

,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss,fold
0,0.5,0.001567,0.998433,0.5,0.001565,0.011755,1
1,0.5,0.000885,0.999115,0.5,0.000885,0.007187,2
2,0.5,0.000646,0.999354,0.5,0.000646,0.005641,3
3,0.5,0.001771,0.998229,0.5,0.001768,0.013159,4
4,0.5,0.001392,0.998608,0.5,0.001390,0.010562,5


### 5.7.3 Summary dummy baselines

In [55]:
# summarise dummy baselines
majority_summary = {
    metric: majority_cv_results[metric].mean()
    for metric in metric_names
}

prior_summary = {
    metric: prior_cv_results[metric].mean()
    for metric in metric_names
}

baseline_results_df = pd.DataFrame([
    {
        "model_name": "dummy_majority",
        **majority_summary
    },
    {
        "model_name": "dummy_prior",
        **prior_summary
    }
])

baseline_results_df

,model_name,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss
0,dummy_majority,0.5,0.001252,0.998748,0.5,0.001252,0.045131
1,dummy_prior,0.5,0.001252,0.998748,0.5,0.001251,0.009661


- Accuracy = 0.999 -> Model predicts no click for everyone but only 0.12% actually click -> Ignore accuracy for clicks
- PR-AUC = 0.001252 -> If I try to find clickers, how good am I. 0.001252 vs 0.12% click rate -> Dummy model has no skill beyond random selection.
- ROC-AUC = 0.5 -> If we randomly pick one clicker and one non-clicker, the probability the model ranks the clicker higher is random guessing (0.5) -> Dummy model cannot distinguish clickers from non-clickers at all.
- Balanced accuracy = 0.5 -> Random guessing -> Model actually has zero discrimination ability.


## 5.8 Build model registry

In [56]:
# define model objects
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

logistic_balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

decision_tree_model = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

decision_tree_balanced_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE
)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest_balanced_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [57]:
# build click model registry
click_model_registry = pd.DataFrame([
    
    {
        "model_name": "logistic_regression",
        "feature_set": "A_core",
        "model_object": logistic_model,
        "notes": "interpretable benchmark"
    },

    {
        "model_name": "logistic_regression",
        "feature_set": "B_interactions",
        "model_object": logistic_model,
        "notes": "tests explicit interactions"
    },

    {
        "model_name": "logistic_regression",
        "feature_set": "C_expanded",
        "model_object": logistic_model,
        "notes": "expanded logistic model"
    },

    {
        "model_name": "logistic_regression_balanced",
        "feature_set": "C_expanded",
        "model_object": logistic_balanced_model,
        "notes": "class-weighted logistic"
    },

    {
        "model_name": "decision_tree",
        "feature_set": "A_core",
        "model_object": decision_tree_model,
        "notes": "simple nonlinear interpretable model"
    },

    {
        "model_name": "decision_tree",
        "feature_set": "C_expanded",
        "model_object": decision_tree_model,
        "notes": "simple nonlinear expanded model"
    },

    {
        "model_name": "decision_tree_balanced",
        "feature_set": "C_expanded",
        "model_object": decision_tree_balanced_model,
        "notes": "class-weighted decision tree"
    },

    {
        "model_name": "random_forest",
        "feature_set": "A_core",
        "model_object": random_forest_model,
        "notes": "stable nonlinear ensemble"
    },

    {
        "model_name": "random_forest",
        "feature_set": "C_expanded",
        "model_object": random_forest_model,
        "notes": "stable nonlinear expanded ensemble"
    },

    {
        "model_name": "random_forest_balanced",
        "feature_set": "C_expanded",
        "model_object": random_forest_balanced_model,
        "notes": "class-weighted random forest"
    }

])

click_model_registry[
    ["model_name", "feature_set", "notes"]
]

,model_name,feature_set,notes
0,logistic_regression,A_core,interpretable benchmark
1,logistic_regression,B_interactions,tests explicit interactions
2,logistic_regression,C_expanded,expanded logistic model
3,logistic_regression_balanced,C_expanded,class-weighted logistic
4,decision_tree,A_core,simple nonlinear interpretable model
5,decision_tree,C_expanded,simple nonlinear expanded model
6,decision_tree_balanced,C_expanded,class-weighted decision tree
7,random_forest,A_core,stable nonlinear ensemble
8,random_forest,C_expanded,stable nonlinear expanded ensemble
9,random_forest_balanced,C_expanded,class-weighted random forest


In [58]:
feature_type_lookup = {
    "A_core": feature_types_A,
    "B_interactions": feature_types_B,
    "C_expanded": feature_types_C
}

In [59]:
for key, value in feature_type_lookup.items():
    print(key)
    print("numeric:", len(value["numeric"]))
    print("categorical:", len(value["categorical"]))
    print("binary:", len(value["binary"]))
    print()

A_core
numeric: 1
categorical: 7
binary: 8

B_interactions
numeric: 1
categorical: 9
binary: 8

C_expanded
numeric: 9
categorical: 7
binary: 12



## 5.9 Build preprocessing + model pipelines

In [ ]:
# build model pipeline factory
def build_model_pipeline(model, feature_set_name):
    """
    Build a sklearn pipeline containing:
    preprocessing recipe for selected feature set + model.
    
    Important:
    The preprocessing step is not fitted here.
    It will be fitted only inside each CV training fold.
    """
    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessors[feature_set_name]),
        ("model", model)
    ])
    
    return pipeline

In [61]:
# build candidate pipeline objects
candidate_pipelines = {}

for _, row in click_model_registry.iterrows():
    model_name = row["model_name"]
    feature_set_name = row["feature_set"]
    model_object = row["model_object"]
    
    candidate_key = f"{model_name}__{feature_set_name}"
    
    candidate_pipelines[candidate_key] = build_model_pipeline(
        model=model_object,
        feature_set_name=feature_set_name
    )

print("Number of candidate pipelines:", len(candidate_pipelines))

for key in candidate_pipelines.keys():
    print(key)

Number of candidate pipelines: 10
logistic_regression__A_core
logistic_regression__B_interactions
logistic_regression__C_expanded
logistic_regression_balanced__C_expanded
decision_tree__A_core
decision_tree__C_expanded
decision_tree_balanced__C_expanded
random_forest__A_core
random_forest__C_expanded
random_forest_balanced__C_expanded


In [62]:
# validate candidate pipelines
for key, pipeline in candidate_pipelines.items():
    assert isinstance(pipeline, Pipeline), f"{key} is not a sklearn Pipeline"
    assert "preprocessing" in pipeline.named_steps, f"{key} missing preprocessing step"
    assert "model" in pipeline.named_steps, f"{key} missing model step"

print("Candidate pipeline validation passed.")

Candidate pipeline validation passed.


## 5.10 Smoke test

### 5.10.1 Normal logistic regression + A_core smoke test

In [63]:
smoke_key = "logistic_regression__A_core"
smoke_model = candidate_pipelines[smoke_key]

X_smoke = trainval_df[feature_sets["A_core"]]
y_smoke = trainval_df[TARGET_COL]
groups_smoke = trainval_df[GROUP_COL]

first_train_idx, first_val_idx = next(
    cv_strategy.split(X_smoke, y_smoke, groups_smoke)
)

X_train_smoke = X_smoke.iloc[first_train_idx]
X_val_smoke = X_smoke.iloc[first_val_idx]

y_train_smoke = y_smoke.iloc[first_train_idx]
y_val_smoke = y_smoke.iloc[first_val_idx]

start_time = time.time()

smoke_model.fit(X_train_smoke, y_train_smoke)

y_pred_smoke = smoke_model.predict(X_val_smoke)
y_proba_smoke = smoke_model.predict_proba(X_val_smoke)[:, 1]

smoke_metrics = calculate_binary_metrics(
    y_true=y_val_smoke,
    y_pred=y_pred_smoke,
    y_proba=y_proba_smoke
)

runtime_seconds = time.time() - start_time

print("Smoke test model:", smoke_key)
print("Smoke test metrics:")
print(smoke_metrics)
print("Predicted positive count:", int(y_pred_smoke.sum()))
print("Mean predicted probability:", y_proba_smoke.mean())
print("Runtime seconds:", round(runtime_seconds, 2))

Smoke test model: logistic_regression__A_core
Smoke test metrics:
{'ROC_AUC': 0.9659215895835196, 'PR_AUC': 0.44335682107500785, 'accuracy': 0.9986100188379026, 'balanced_accuracy': 0.5680750672880854, 'brier': 0.001133873140978338, 'log_loss': 0.00508287509530057}
Predicted positive count: 41
Mean predicted probability: 0.001308803115861859
Runtime seconds: 5.03


### 5.10.2 Balanced logistic regression + C_expanded smoke test

In [64]:
smoke_balanced_key = "logistic_regression_balanced__C_expanded"
smoke_balanced_model = candidate_pipelines[smoke_balanced_key]

X_smoke_balanced = trainval_df[feature_sets["C_expanded"]]
y_smoke_balanced = trainval_df[TARGET_COL]
groups_smoke_balanced = trainval_df[GROUP_COL]

first_train_idx_bal, first_val_idx_bal = next(
    cv_strategy.split(X_smoke_balanced, y_smoke_balanced, groups_smoke_balanced)
)

X_train_smoke_bal = X_smoke_balanced.iloc[first_train_idx_bal]
X_val_smoke_bal = X_smoke_balanced.iloc[first_val_idx_bal]

y_train_smoke_bal = y_smoke_balanced.iloc[first_train_idx_bal]
y_val_smoke_bal = y_smoke_balanced.iloc[first_val_idx_bal]

start_time = time.time()

smoke_balanced_model.fit(X_train_smoke_bal, y_train_smoke_bal)

y_pred_smoke_bal = smoke_balanced_model.predict(X_val_smoke_bal)
y_proba_smoke_bal = smoke_balanced_model.predict_proba(X_val_smoke_bal)[:, 1]

smoke_balanced_metrics = calculate_binary_metrics(
    y_true=y_val_smoke_bal,
    y_pred=y_pred_smoke_bal,
    y_proba=y_proba_smoke_bal
)

runtime_seconds_bal = time.time() - start_time

print("Smoke test model:", smoke_balanced_key)
print("Smoke test metrics:")
print(smoke_balanced_metrics)
print("Predicted positive count:", int(y_pred_smoke_bal.sum()))
print("Mean predicted probability:", y_proba_smoke_bal.mean())
print("Runtime seconds:", round(runtime_seconds_bal, 2))

Smoke test model: logistic_regression_balanced__C_expanded
Smoke test metrics:
{'ROC_AUC': 0.9364167922588967, 'PR_AUC': 0.37580062700893635, 'accuracy': 0.9795831275795429, 'balanced_accuracy': 0.896536873747295, 'brier': 0.027675086903296037, 'log_loss': 0.136436428640214}
Predicted positive count: 3510
Mean predicted probability: 0.0985749228546166
Runtime seconds: 27.04


Smoke test passed.

Model works.

Metrics are not bad.

But performance is high enough that we should keep leakage/sanity checks in mind.

### 5.10.3 Data leakage and sanity checks

In [65]:
# feature leakage audit
forbidden_feature_cols = [
    "click", "open",
    "user_id", "mailing_id",
    "mailing_info", "subject_line", "preheader",
    "send_date", "send_day_of_week", "date_trusted"
]

for name, features in feature_sets.items():
    forbidden_found = [col for col in features if col in forbidden_feature_cols]
    print(name, forbidden_found)

A_core []
B_interactions []
C_expanded []


In [66]:
# sanity check feature set without direct prior-click features
direct_prior_click_features = [
    "had_prior_click",
    "prior_click_count",
    "historical_click_rate"
]

feature_set_C_no_prior_click = [
    col for col in feature_set_C_expanded
    if col not in direct_prior_click_features
]

print("Original C features:", len(feature_set_C_expanded))
print("C without prior-click features:", len(feature_set_C_no_prior_click))
print("Removed:", direct_prior_click_features)

Original C features: 28
C without prior-click features: 25
Removed: ['had_prior_click', 'prior_click_count', 'historical_click_rate']


In [68]:
# define feature types for feature set without direct prior-click features
feature_types_C_no_prior_click = {
    "numeric": [
        col for col in feature_types_C["numeric"]
        if col not in direct_prior_click_features
    ],
    "categorical": feature_types_C["categorical"].copy(),
    "binary": [
        col for col in feature_types_C["binary"]
        if col not in direct_prior_click_features
    ]
}

preprocessor_C_no_prior_click = build_preprocessor(feature_types_C_no_prior_click)

In [69]:
# logistic C without direct prior-click features smoke test
sanity_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor_C_no_prior_click),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

X_sanity = trainval_df[feature_set_C_no_prior_click]
y_sanity = trainval_df[TARGET_COL]
groups_sanity = trainval_df[GROUP_COL]

first_train_idx, first_val_idx = next(
    cv_strategy.split(X_sanity, y_sanity, groups_sanity)
)

X_train_sanity = X_sanity.iloc[first_train_idx]
X_val_sanity = X_sanity.iloc[first_val_idx]

y_train_sanity = y_sanity.iloc[first_train_idx]
y_val_sanity = y_sanity.iloc[first_val_idx]

sanity_pipeline.fit(X_train_sanity, y_train_sanity)

y_pred_sanity = sanity_pipeline.predict(X_val_sanity)
y_proba_sanity = sanity_pipeline.predict_proba(X_val_sanity)[:, 1]

sanity_metrics = calculate_binary_metrics(
    y_true=y_val_sanity,
    y_pred=y_pred_sanity,
    y_proba=y_proba_sanity
)

print("Sanity check: Logistic C without direct prior-click features")
print(sanity_metrics)
print("Predicted positive count:", int(y_pred_sanity.sum()))
print("Mean predicted probability:", y_proba_sanity.mean())

Sanity check: Logistic C without direct prior-click features
{'ROC_AUC': 0.6874324986805629, 'PR_AUC': 0.030056786530219377, 'accuracy': 0.9984332229883376, 'balanced_accuracy': 0.5, 'brier': 0.001549261244431732, 'log_loss': 0.01108972790052837}
Predicted positive count: 0
Mean predicted probability: 0.0015095349940383185


The huge click performance is mostly coming from direct prior-click history, but even without prior-click features, the model still has some useful signal.

High click performance is not automatically leakage; it is mainly explained by prior click behaviour. We should document this and later maybe report a robustness model without prior-click features.

## 5.11 Run broad CV screening

In [70]:
# evaluate one click candidate model with CV
def evaluate_candidate_cv(row):
    model_name = row["model_name"]
    feature_set_name = row["feature_set"]
    notes = row["notes"]
    
    candidate_key = f"{model_name}__{feature_set_name}"
    pipeline = candidate_pipelines[candidate_key]
    
    X = trainval_df[feature_sets[feature_set_name]]
    y = trainval_df[TARGET_COL]
    groups = trainval_df[GROUP_COL]
    
    start_time = time.time()
    
    fold_results = evaluate_cv_model(
        model=pipeline,
        X=X,
        y=y,
        groups=groups,
        cv_strategy=cv_strategy,
        metric_names=metric_names
    )
    
    runtime_seconds = time.time() - start_time
    
    summary = {
        "target": TARGET_COL,
        "model_name": model_name,
        "feature_set": feature_set_name,
        "candidate_key": candidate_key,
        "notes": notes,
        "runtime_seconds": runtime_seconds
    }
    
    for metric in metric_names:
        summary[f"mean_{metric}"] = fold_results[metric].mean()
        summary[f"std_{metric}"] = fold_results[metric].std()
    
    return summary, fold_results

In [71]:
# run broad CV screening for click
screening_summaries = []
screening_fold_results = {}

for i, row in click_model_registry.iterrows():
    candidate_key = f"{row['model_name']}__{row['feature_set']}"
    
    print("=" * 80)
    print(f"Running candidate {i + 1}/{len(click_model_registry)}: {candidate_key}")
    
    try:
        summary, fold_results = evaluate_candidate_cv(row)
        screening_summaries.append(summary)
        screening_fold_results[candidate_key] = fold_results
        
        print("Done.")
        print("Mean PR-AUC:", round(summary["mean_PR_AUC"], 6))
        print("Mean ROC-AUC:", round(summary["mean_ROC_AUC"], 4))
        print("Mean predicted? See fold results if needed.")
        print("Runtime seconds:", round(summary["runtime_seconds"], 2))
        
    except Exception as e:
        print("FAILED:", candidate_key)
        print("Error:", e)

Running candidate 1/10: logistic_regression__A_core
Done.
Mean PR-AUC: 0.319062
Mean ROC-AUC: 0.9505
Mean predicted? See fold results if needed.
Runtime seconds: 32.91
Running candidate 2/10: logistic_regression__B_interactions
Done.
Mean PR-AUC: 0.318563
Mean ROC-AUC: 0.9518
Mean predicted? See fold results if needed.
Runtime seconds: 20.75
Running candidate 3/10: logistic_regression__C_expanded
Done.
Mean PR-AUC: 0.386815
Mean ROC-AUC: 0.9563
Mean predicted? See fold results if needed.
Runtime seconds: 38.35
Running candidate 4/10: logistic_regression_balanced__C_expanded
Done.
Mean PR-AUC: 0.353431
Mean ROC-AUC: 0.9424
Mean predicted? See fold results if needed.
Runtime seconds: 104.26
Running candidate 5/10: decision_tree__A_core
Done.
Mean PR-AUC: 0.161292
Mean ROC-AUC: 0.7086
Mean predicted? See fold results if needed.
Runtime seconds: 31.03
Running candidate 6/10: decision_tree__C_expanded
Done.
Mean PR-AUC: 0.193967
Mean ROC-AUC: 0.726
Mean predicted? See fold results if needed

In [72]:
# create click screening results table
screening_results_df = pd.DataFrame(screening_summaries)

screening_results_df = screening_results_df.sort_values(
    by="mean_PR_AUC",
    ascending=False
).reset_index(drop=True)

display(screening_results_df)

,target,model_name,feature_set,candidate_key,notes,runtime_seconds,mean_ROC_AUC,std_ROC_AUC,mean_PR_AUC,std_PR_AUC,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,mean_brier,std_brier,mean_log_loss,std_log_loss
0,click,random_forest,C_expanded,random_forest__C_expanded,stable nonlinear expanded ensemble,95.732967,0.927475,0.022739,0.508435,0.088197,0.999028,0.000318,0.649010,0.021786,0.000815,0.000220,0.009183,0.004519
1,click,random_forest_balanced,C_expanded,random_forest_balanced__C_expanded,class-weighted random forest,89.104418,0.919088,0.026233,0.447213,0.092001,0.998909,0.000362,0.590378,0.023875,0.000876,0.000256,0.010217,0.005324
2,click,logistic_regression,C_expanded,logistic_regression__C_expanded,expanded logistic model,38.351653,0.956329,0.016601,0.386815,0.102058,0.998809,0.000395,0.612044,0.023670,0.000947,0.000284,0.004403,0.001547
3,click,random_forest,A_core,random_forest__A_core,stable nonlinear ensemble,100.850984,0.916744,0.019652,0.360430,0.117603,0.998866,0.000323,0.597736,0.026678,0.000962,0.000256,0.010550,0.004464
4,click,logistic_regression_balanced,C_expanded,logistic_regression_balanced__C_expanded,class-weighted logistic,104.264547,0.942382,0.014207,0.353431,0.087197,0.950122,0.036043,0.908582,0.028609,0.044087,0.023586,0.175385,0.061835
5,click,logistic_regression,A_core,logistic_regression__A_core,interpretable benchmark,32.914503,0.950452,0.015845,0.319062,0.117017,0.998731,0.000405,0.567836,0.031625,0.001022,0.000308,0.004660,0.001568
6,click,logistic_regression,B_interactions,logistic_regression__B_interactions,tests explicit interactions,20.754242,0.951785,0.018498,0.318563,0.126030,0.998742,0.000429,0.566624,0.035514,0.001018,0.000318,0.004666,0.001635
7,click,decision_tree,C_expanded,decision_tree__C_expanded,simple nonlinear expanded model,45.523842,0.725973,0.035704,0.193967,0.088585,0.998129,0.000773,0.726108,0.035636,0.001752,0.000468,0.054831,0.005352
8,click,decision_tree,A_core,decision_tree__A_core,simple nonlinear interpretable model,31.025461,0.708614,0.033289,0.161292,0.072962,0.998295,0.000171,0.706950,0.032031,0.001758,0.000186,0.059328,0.004845
9,click,decision_tree_balanced,C_expanded,decision_tree_balanced__C_expanded,class-weighted decision tree,40.067780,0.665881,0.031648,0.143465,0.057860,0.998394,0.000152,0.665852,0.031682,0.001599,0.000144,0.050981,0.008975


In [73]:
# compact click screening leaderboard
leaderboard_cols = [
    "model_name",
    "feature_set",
    "mean_PR_AUC",
    "std_PR_AUC",
    "mean_ROC_AUC",
    "std_ROC_AUC",
    "mean_accuracy",
    "mean_balanced_accuracy",
    "mean_brier",
    "mean_log_loss",
    "runtime_seconds",
    "notes"
]

display(screening_results_df[leaderboard_cols])

,model_name,feature_set,mean_PR_AUC,std_PR_AUC,mean_ROC_AUC,std_ROC_AUC,mean_accuracy,mean_balanced_accuracy,mean_brier,mean_log_loss,runtime_seconds,notes
0,random_forest,C_expanded,0.508435,0.088197,0.927475,0.022739,0.999028,0.649010,0.000815,0.009183,95.732967,stable nonlinear expanded ensemble
1,random_forest_balanced,C_expanded,0.447213,0.092001,0.919088,0.026233,0.998909,0.590378,0.000876,0.010217,89.104418,class-weighted random forest
2,logistic_regression,C_expanded,0.386815,0.102058,0.956329,0.016601,0.998809,0.612044,0.000947,0.004403,38.351653,expanded logistic model
3,random_forest,A_core,0.360430,0.117603,0.916744,0.019652,0.998866,0.597736,0.000962,0.010550,100.850984,stable nonlinear ensemble
4,logistic_regression_balanced,C_expanded,0.353431,0.087197,0.942382,0.014207,0.950122,0.908582,0.044087,0.175385,104.264547,class-weighted logistic
5,logistic_regression,A_core,0.319062,0.117017,0.950452,0.015845,0.998731,0.567836,0.001022,0.004660,32.914503,interpretable benchmark
6,logistic_regression,B_interactions,0.318563,0.126030,0.951785,0.018498,0.998742,0.566624,0.001018,0.004666,20.754242,tests explicit interactions
7,decision_tree,C_expanded,0.193967,0.088585,0.725973,0.035704,0.998129,0.726108,0.001752,0.054831,45.523842,simple nonlinear expanded model
8,decision_tree,A_core,0.161292,0.072962,0.708614,0.033289,0.998295,0.706950,0.001758,0.059328,31.025461,simple nonlinear interpretable model
9,decision_tree_balanced,C_expanded,0.143465,0.057860,0.665881,0.031648,0.998394,0.665852,0.001599,0.050981,40.067780,class-weighted decision tree


Click behaviour is driven largely by historical engagement patterns that can already be captured reasonably well by logistic regression. Nonlinear models provide additional predictive gains, with random forests achieving the strongest performance, suggesting the presence of interaction and threshold effects that are not fully captured by linear models. Decision trees alone are less effective because the extreme class imbalance and sparse positive class make it difficult for a single tree to learn stable click patterns.

## 5.12 Save Stage 5 outputs

In [77]:
# define Stage 5 click output paths
BASELINE_RESULTS_PATH = OUTPUT_DIR / "5_baseline_results_click_v1.csv"
SCREENING_RESULTS_PATH = OUTPUT_DIR / "5_screening_results_click_v1.csv"
CV_FOLD_SUMMARY_PATH = OUTPUT_DIR / "5_cv_fold_summary_click_v1.csv"
CANDIDATE_SPECS_PATH = OUTPUT_DIR / "5_candidate_specs_click_v1.csv"
STAGE5_NOTES_PATH = OUTPUT_DIR / "5_modelling_stage5_notes_click_v1.md"

print(BASELINE_RESULTS_PATH)
print(SCREENING_RESULTS_PATH)
print(CV_FOLD_SUMMARY_PATH)
print(CANDIDATE_SPECS_PATH)
print(STAGE5_NOTES_PATH)

/Users/khanhhien/Documents/Thesis/Outputs/5_baseline_results_click_v1.csv
/Users/khanhhien/Documents/Thesis/Outputs/5_screening_results_click_v1.csv
/Users/khanhhien/Documents/Thesis/Outputs/5_cv_fold_summary_click_v1.csv
/Users/khanhhien/Documents/Thesis/Outputs/5_candidate_specs_click_v1.csv
/Users/khanhhien/Documents/Thesis/Outputs/5_modelling_stage5_notes_click_v1.md


In [78]:
# save Stage 5 click CSV outputs
baseline_results_df.to_csv(BASELINE_RESULTS_PATH, index=False)
screening_results_df.to_csv(SCREENING_RESULTS_PATH, index=False)
cv_fold_summary_click.to_csv(CV_FOLD_SUMMARY_PATH, index=False)

candidate_specs_click_df = click_model_registry[
    ["model_name", "feature_set", "notes"]
].copy()

candidate_specs_click_df.to_csv(CANDIDATE_SPECS_PATH, index=False)

print("Saved baseline results:", BASELINE_RESULTS_PATH.exists())
print("Saved screening results:", SCREENING_RESULTS_PATH.exists())
print("Saved CV fold summary:", CV_FOLD_SUMMARY_PATH.exists())
print("Saved candidate specs:", CANDIDATE_SPECS_PATH.exists())

Saved baseline results: True
Saved screening results: True
Saved CV fold summary: True
Saved candidate specs: True


In [79]:
# save Stage 5 click notes
stage5_notes = f"""
# Modelling Stage 5 Notes — Click Prediction Screening v1

## Purpose

Stage 5 is the first actual modelling stage for click prediction. The purpose is to turn the frozen click dataset and preprocessing design into working sklearn model pipelines, run dummy baselines, run broad candidate screening, and produce the first cross-validation comparison table.

This stage does not perform deep hyperparameter tuning, final test evaluation, SHAP interpretation, or final model selection.

## Dataset

Input dataset:

- df_click_model_v1.parquet

Target:

- {TARGET_COL}

Group column:

- {GROUP_COL}

Unit of analysis:

- one user-campaign observation

## Important leakage rule

This is pre-send click prediction.

Current open is a post-send outcome and was excluded as a predictor. The target click itself, current open, user_id, mailing_id, raw text fields, and audit/date fields were excluded from feature sets.

## Split design

The data was split by campaign using mailing_id as the chronological proxy.

- Earliest 80% campaigns: train-validation pool
- Latest 20% campaigns: final untouched test set

Final test was created but not used for model selection or performance reporting in Stage 5.

## Train-validation / test summary

{split_summary_click.to_markdown(index=False)}

## Cross-validation

Inner CV strategy:

- StratifiedGroupKFold
- n_splits = {INNER_CV_SPLITS}
- shuffle = True
- random_state = {RANDOM_STATE}
- groups = mailing_id

Fold quality checks passed:

- no campaign overlap between train and validation fold
- each validation fold contains both click and non-click observations

## Feature sets

Feature sets used:

- A_core: clean thesis-aligned interpretable feature set
- B_interactions: A_core plus two explicit interaction features
- C_expanded: broader predictive feature set with raw counts/rates and additional content indicators

Feature counts:

- A_core: {len(feature_set_A_core)}
- B_interactions: {len(feature_set_B_interactions)}
- C_expanded: {len(feature_set_C_expanded)}

Interaction features in B:

- campaign_topic_x_age_group
- historical_open_bucket_x_prior_email_frequency_bucket

## Baselines

Two dummy baselines were evaluated:

- dummy_majority: predicts the most frequent class, i.e. no click
- dummy_prior: predicts the training class prior probability

Baseline results:

{baseline_results_df.to_markdown(index=False)}

## Candidate models screened

Candidate models:

{candidate_specs_click_df.to_markdown(index=False)}

All real models were wrapped as sklearn pipelines:

preprocessing + model

This ensures that imputation, scaling, and one-hot encoding are fitted only inside each CV training fold.

## Screening results

Main screening leaderboard, sorted by mean PR-AUC:

{screening_results_df[leaderboard_cols].to_markdown(index=False)}

## Main early observations

1. Click prediction is an extreme rare-event task.
2. Accuracy is not meaningful as a headline metric because the no-click class dominates.
3. PR-AUC was used as the primary screening metric.
4. All real models clearly outperform dummy baselines by PR-AUC.
5. C_expanded improves performance substantially, especially for logistic regression and random forest.
6. Random forest with C_expanded achieved the strongest screening performance.
7. Class-weighted models improved some threshold-style behaviour, especially balanced accuracy, but did not necessarily improve PR-AUC.
8. Decision trees underperformed logistic regression and random forest, likely because a single tree struggles to learn stable patterns under extreme class imbalance.
9. The very strong click performance appears largely driven by prior-click history features.

## Sanity check

A no-prior-click sanity check was run by removing:

- had_prior_click
- prior_click_count
- historical_click_rate

The reduced logistic model still exceeded the dummy baseline, but performance dropped sharply compared with the full model. This suggests that the strong full-model performance is mainly explained by prior click behaviour rather than obvious leakage.

Sanity check result from first fold:

- ROC-AUC: {sanity_metrics["ROC_AUC"]}
- PR-AUC: {sanity_metrics["PR_AUC"]}
- accuracy: {sanity_metrics["accuracy"]}
- balanced accuracy: {sanity_metrics["balanced_accuracy"]}
- brier: {sanity_metrics["brier"]}
- log loss: {sanity_metrics["log_loss"]}

## XGBoost status

XGBoost was not available in the current Python environment during Stage 5. It was not included in this screening run.

XGBoost may be added later as an optional boosted-tree extension after the current screening outputs are saved and reviewed.

## Temporal feature status

Temporal variables were not included in A/B/C feature sets during Stage 5.

If temporal features are added later, they should be introduced as a documented new feature set, for example D_temporal, rather than silently modifying A/B/C.

## Next step

Proceed to Stage 6:

- shortlist click candidates for tuning
- likely keep logistic regression C_expanded as the main interpretable click benchmark
- likely keep random forest C_expanded as the strongest predictive candidate
- consider whether the no-prior-click robustness model should be formally included later
- consider whether XGBoost should be installed and added as an optional comparison
- do not use the final test set until final model evaluation
"""

with open(STAGE5_NOTES_PATH, "w", encoding="utf-8") as f:
    f.write(stage5_notes)

print("Saved Stage 5 click notes:", STAGE5_NOTES_PATH.exists())

Saved Stage 5 click notes: True
